# exp_validate_clf — does training on the frozen R actually rank better?

Scores clf-v4 (old) vs `clf_R` (retrained on `elig_first-L512`) by **judged-pool NDCG@10**, both under
the frozen representation. TREC21 is comparable to the truncation study's 0.8933 (in-sample for both);
TREC22 is the clean held-out. macro-F1 doesn't answer this — ranking does.


## Setup (Colab — GPU for cross-encoder scoring; judged pool only, so light)


In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q transformers datasets pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from ctmatch.experiments import (ExperimentConfig, load_corpus, load_eval,
                                 cross_encoder_scores, relevant_index, ndcg_at_k)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = ExperimentConfig(data_root=DATA_ROOT)   # frozen elig_first-L512
print('repr:', cfg.repr_tag())


In [ ]:
# Corpus fields + judged pools (no retrieval). Keep only judged docs present in the corpus
# (some judged NCT IDs aren't in the 2021 snapshot -> can't be scored).
corpus_ids, corpus_fields = load_corpus(cfg)
id2fields = dict(zip(corpus_ids, corpus_fields))
sets = load_eval(cfg, ['trec21', 'trec22'])
pools = {s: {t: [d for d in sets[s]['rel_dict'][t] if d in id2fields]
             for t in sets[s]['rel_dict'] if t in sets[s]['topic2text']}
         for s in sets}
print({s: len(p) for s, p in pools.items()}, 'topics')


In [ ]:
# Score each checkpoint on each pool under the frozen R; judged-pool NDCG@10.
CKPTS = {'clf-v4 (old)': 'semaj83/ctmatch-clf-v4', 'clf_R (frozen R)': cfg.path('models/clf_R')}

def pool_ndcg(model, tok, split):
    REL = relevant_index(model)
    rel, topic2text = sets[split]['rel_dict'], sets[split]['topic2text']
    vals = []
    for t, docs in pools[split].items():
        if not docs:
            continue
        s = cross_encoder_scores(model, tok, topic2text[t], [id2fields[d] for d in docs], cfg, REL)
        ranked = [d for d, _ in sorted(zip(docs, s), key=lambda x: -x[1])]
        vals.append(ndcg_at_k(ranked, rel[t]))
    return float(np.mean(vals))

rows = []
for name, ckpt in CKPTS.items():
    tok = AutoTokenizer.from_pretrained(ckpt)
    model = AutoModelForSequenceClassification.from_pretrained(ckpt).to(device).eval()
    rows.append({'model': name, **{f'{s}_ndcg@10': round(pool_ndcg(model, tok, s), 4) for s in pools}})
    del model; torch.cuda.empty_cache()
pd.DataFrame(rows)


## Reading it
- **TREC22 (held-out) is the honest comparison** — both models are blind to it. If `clf_R` ≥ clf-v4 there,
  training on the frozen representation held up (and the earlier 0.893 wasn't just a distribution-match
  artifact). If lower, part of clf-v4's edge was that unfair match, and this is the true baseline.
- TREC21 is in-sample for both, so read it only as a consistency check vs the study's 0.8933.
- Then bump `cfg.clf_ckpt` to `clf_R` (push it) so every downstream stage uses the frozen-R model.
